# MCAI500 M3 In Class Activity

Module 3 focuses on classification methods and reliable evaluation.

# In-class activity

Work in pairs. Complete the missing parts, run the experiments, and write short interpretations in Markdown cells. The goal is not only to obtain a score; it is to explain why the score and predictions change.

## Activity plan

1. Load and inspect the data.
2. Complete the model pipelines.
3. Compare at least three algorithms.
4. Change hyperparameters and record what changes.
5. Interpret a confusion matrix and ROC curve.
6. Decide which model is most appropriate for detecting malignant cases.

In [2]:
# Core tools used throughout the notebook.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    ConfusionMatrixDisplay, make_scorer
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 12)
sns.set_theme(style="whitegrid")


## Task 1: Load the data and define the positive class

**Hint:** Use `load_breast_cancer(as_frame=True)`. The original target is 0 for malignant and 1 for benign. Recode it so malignant is 1. Then create `X` and `y`.

Complete the code below and check the class counts.

In [3]:
dataset = load_breast_cancer(as_frame=True)
df = dataset.frame.copy()
# TODO: Recode the target so malignant = 1 and benign = 0.
df["target"] = (df["target"] == 0).astype(int)
df["target_label"] = df["target"].map({0: "benign", 1: "malignant"})

# TODO: Create X by removing the target column and create y.
X = df.drop(columns=["target", "target_label"])
y = df["target"]


print("Dataset shape:", df.shape)
display(df.head())
display(y.value_counts().rename(index={0: "benign", 1: "malignant"}).to_frame("count"))


Dataset shape: (569, 32)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,...,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target,target_label
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,...,0.7119,0.2654,0.4601,0.11890,1,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,...,0.2416,0.1860,0.2750,0.08902,1,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,...,0.4504,0.2430,0.3613,0.08758,1,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,...,0.6869,0.2575,0.6638,0.17300,1,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,...,0.4000,0.1625,0.2364,0.07678,1,malignant


,count
target,
benign,357
malignant,212


## Task 2: Create a stratified train/test split

Use 20% for testing and `random_state=42`. **Hint:** pass `stratify=y` so that both sets keep similar class proportions.

In [4]:
# TODO: Complete the split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Training set shape: (455, 30)
Test set shape: (114, 30)


## Task 3: Build three classifiers

Complete the pipelines. Use scaling for Logistic Regression and KNN. Random Forest does not require scaling.

Suggested starting values:
- Logistic Regression: `max_iter=2000`
- KNN: `n_neighbors=7`
- Random Forest: `n_estimators=250`, `max_depth=6`

**Question:** Why might scaling matter for KNN?

In [8]:
# TODO: Define the three models.
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=7))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(n_estimators=250,
                                         max_depth=6, 
                                           random_state=RANDOM_STATE,
                                           n_jobs=-1))
    ])
}

## Task 4: Compare metrics

For each model, fit on the training data and calculate accuracy, precision, recall, and F1 on the test data.

**Hint:** Use `model.predict(X_test)`, then `accuracy_score`, `precision_score`, `recall_score`, and `f1_score`.

Remember: the positive class is malignant (1).

In [10]:
rows = []
predictions = {}

for name, model in models.items():
    # Treinar o modelo
    model.fit(X_train, y_train)

    # Prever as classes dos dados de teste
    y_pred = model.predict(X_test)

    # Guardar as previsões para análises posteriores
    predictions[name] = y_pred

    # Comparar as previsões com as respostas corretas
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0)
    })

comparison = pd.DataFrame(rows).set_index("Model")
display(comparison.round(3))

# TODO: Complete the loop and create a comparison DataFrame.
# rows = []
# predictions = {}
# for name, model in models.items():
#     model.fit(X_train, y_train)
#     y_pred = model.predict(X_test)
#     predictions[name] = y_pred
#     rows.append({
#         "Model": name,
#         "Accuracy": ...,
#         "Precision": ...,
#         "Recall": ...,
#         "F1": ...
#     })
# comparison = pd.DataFrame(rows).set_index("Model")
# display(comparison.round(3))

,Accuracy,Precision,Recall,F1
Model,,,,
Logistic Regression,0.965,0.975,0.929,0.951
KNN,0.956,0.974,0.905,0.938
Random Forest,0.965,1.000,0.905,0.950


## Task 5: Change model hyperparameters

Run at least two experiments:

- Change KNN `n_neighbors` to 1, 5, 15, and  thirty.
- Change Random Forest `max_depth` to 2, 6, and `None`.
- Change the number of trees to 50 and 300.

Record the test F1 and recall. Do not assume that the largest model is automatically best.

In [ ]:
# TODO: Choose one experiment and record the result for each value.
# For example, vary KNN: [1, 5, 15, 30].
# experiment_rows = []
# for k in [1, 5, 15, 30]:
#     candidate = Pipeline([
#         ("scaler", StandardScaler()),
#         ("model", KNeighborsClassifier(n_neighbors=k))
#     ])
#     candidate.fit(X_train, y_train)
#     pred = candidate.predict(X_test)
#     experiment_rows.append({"k": k, "F1": ..., "Recall": ...})
# display(pd.DataFrame(experiment_rows))

## Task 6: Confusion matrix and error costs

Choose the model with the highest recall, or explain why you choose another one. Plot its confusion matrix.

Then answer: is a false negative or a false positive more concerning in this medical example?

In [ ]:
# TODO: Select a model and plot its confusion matrix.
# chosen_name = ...
# ConfusionMatrixDisplay.from_predictions(
#     y_test, predictions[chosen_name],
#     display_labels=["Benign (0)", "Malignant (1)"], cmap="Blues"
# )
# plt.show()

## Task 7: Cross-validation

Use `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` and `cross_validate` or `cross_val_score`. Compare the mean and standard deviation of F1 or ROC-AUC.

**Hint:** Put preprocessing inside the pipeline so each fold learns its scaling only from its training portion.

In [ ]:
# TODO: Calculate 5-fold cross-validated F1 for each model.
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# for name, model in models.items():
#     scores = cross_validate(model, X_train, y_train, cv=cv, scoring="f1")
#     print(name, scores["test_score"].mean(), scores["test_score"].std())

## Task 8: ROC curve and threshold experiment

Use `predict_proba` to obtain malignant probabilities. Plot an ROC curve for at least two models. Then compare thresholds 0.30, 0.50, and 0.70 for Logistic Regression.

**Questions:**
- Which threshold gives the highest recall?
- What happens to precision?
- Which threshold would you choose for screening, and why?

In [ ]:
# TODO: Fit Logistic Regression and obtain probability scores.
# log_model = models["Logistic Regression"]
# log_model.fit(X_train, y_train)
# probabilities = log_model.predict_proba(X_test)[:, 1]

# TODO: For each threshold in [0.30, 0.50, 0.70], calculate precision, recall, and F1.
# for threshold in [...]:
#     threshold_pred = (probabilities >= threshold).astype(int)
#     print(threshold, precision_score(...), recall_score(...), f1_score(...))

## Final student response

Write a short conclusion:

1. Which model performed best according to your chosen metric?
2. Which errors did it make?
3. How did the hyperparameters affect the result?
4. Why can different algorithms produce different predictions on the same data?